# CIFAR-10 residual network under 5M parameters
Run top to bottom on Kaggle with a GPU. Attach the competition data. Paths below point to `train_labels.csv` and `train_images` in the competition directory. No pretrained weights or external training data are used.

The fixed, stratified 80/20 split is saved for reproducibility. Training uses random crops and horizontal flips; validation uses deterministic normalization. The small-batch overfit check has passed on Kaggle. This notebook now trains from scratch for 150 epochs and saves the checkpoint with the highest validation accuracy.


In [ ]:
from pathlib import Path
import csv
import random
import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

SEED = 42
BATCH_SIZE = 128
EPOCHS = 150
LEARNING_RATE = 0.1
NUM_WORKERS = 2
INPUT_ROOT = Path('/kaggle/input/competitions/cifar-10-res-net-architecture-challenge-dl')
WORK_DIR = Path('/kaggle/working')
WORK_DIR.mkdir(parents=True, exist_ok=True)
LABELS_CSV = INPUT_ROOT / 'train_labels.csv'
TRAIN_IMAGE_DIR = INPUT_ROOT / 'train_images'

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


## Load labels and freeze the split
Expected data: a CSV with `id,label` columns and one image per ID (for example `train_images/train_00001.png`). Class names are mapped in sorted order and saved. The split uses a dedicated NumPy generator, independent of training randomness. Each class must have a sample count divisible by five so the requested 80/20 ratio is exact within every class.


In [ ]:
assert LABELS_CSV.is_file(), f'Missing labels CSV: {LABELS_CSV}'
assert TRAIN_IMAGE_DIR.is_dir(), f'Missing training image directory: {TRAIN_IMAGE_DIR}'
LABELS_CSV = Path(LABELS_CSV)
with LABELS_CSV.open(newline='') as f:
    reader = csv.DictReader(f)
    assert {'id', 'label'} <= set(reader.fieldnames or []), 'Expected id,label columns'
    rows = sorted(list(reader), key=lambda row: row['id'])
assert rows and len({r['id'] for r in rows}) == len(rows), 'IDs must be unique'
classes = sorted({r['label'] for r in rows})
assert len(classes) == 10, classes
class_to_idx = {name: i for i, name in enumerate(classes)}
labels = np.array([class_to_idx[r['label']] for r in rows], dtype=np.int64)
TRAIN_IMAGE_DIR = Path(TRAIN_IMAGE_DIR)
def training_image_path(image_id):
    # Accept numeric IDs, train_00001 IDs, or train_00001.png filenames.
    stem = Path(str(image_id).strip()).stem
    if stem.startswith('train_'):
        stem = stem[len('train_'):]
    if not stem.isdigit():
        raise ValueError(f'Unexpected training image ID: {image_id!r}')
    return TRAIN_IMAGE_DIR / f'train_{int(stem):05d}.png'

image_paths = [training_image_path(r['id']) for r in rows]
missing = [str(p) for p in image_paths if not p.is_file()]
assert not missing, f'Missing images, first examples: {missing[:5]}'

rng = np.random.default_rng(SEED)
train_indices, val_indices = [], []
for cls in range(10):
    indices = rng.permutation(np.flatnonzero(labels == cls))
    assert len(indices) % 5 == 0, 'Exact stratified 80/20 requires class sizes divisible by 5'
    n_val = len(indices) // 5
    val_indices.extend(indices[:n_val].tolist())
    train_indices.extend(indices[n_val:].tolist())
train_indices = np.array(sorted(train_indices))
val_indices = np.array(sorted(val_indices))
assert len(train_indices) * 5 == len(rows) * 4
assert len(val_indices) * 5 == len(rows)
assert not set(train_indices) & set(val_indices)
assert set(train_indices) | set(val_indices) == set(range(len(rows)))
with (WORK_DIR / 'split.csv').open('w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'label', 'split'])
    val_set = set(val_indices)
    for i, row in enumerate(rows):
        writer.writerow([row['id'], row['label'], 'validation' if i in val_set else 'training'])
print('Class mapping:', class_to_idx)
print('Training / validation:', len(train_indices), len(val_indices))
for cls, name in enumerate(classes):
    print(name, int((labels[train_indices] == cls).sum()), int((labels[val_indices] == cls).sum()))

In [ ]:
# Fixed CIFAR normalization constants; no statistics are fitted on validation data.
MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class LabeledImages(Dataset):
    def __init__(self, indices, transform):
        self.indices = np.asarray(indices)
        self.transform = transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, index):
        i = int(self.indices[index])
        with Image.open(image_paths[i]) as image:
            image = image.convert('RGB')
            assert image.size == (32, 32), f'Unexpected image size: {image.size}'
            x = self.transform(image)
        return x, int(labels[i])

train_dataset = LabeledImages(train_indices, train_transform)
val_dataset = LabeledImages(val_indices, val_transform)
assert torch.equal(val_dataset[0][0], val_dataset[0][0]), 'Validation must be deterministic'

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# A dedicated generator fixes training shuffle order across fresh runs.
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, worker_init_fn=seed_worker,
    generator=torch.Generator().manual_seed(SEED),
    pin_memory=device.type == 'cuda',
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=device.type == 'cuda',
)


## Three-stage basic-block ResNet
The stem is a 3×3 stride-1 convolution with no max pooling. Stages have 64, 128, and 256 channels and three residual blocks each. Only the first blocks of stages 2 and 3 downsample, using stride 2 and a learned 1×1 shortcut. Global average pooling feeds a ten-class linear classifier.


In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.residual = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.shortcut = nn.Identity() if stride == 1 and in_channels == out_channels else nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.relu(self.residual(x) + self.shortcut(x))

class CIFARResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.stage1 = nn.Sequential(BasicBlock(64, 64), BasicBlock(64, 64), BasicBlock(64, 64))
        self.stage2 = nn.Sequential(BasicBlock(64, 128, 2), BasicBlock(128, 128), BasicBlock(128, 128))
        self.stage3 = nn.Sequential(BasicBlock(128, 256, 2), BasicBlock(256, 256), BasicBlock(256, 256))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(256, 10)
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
    def forward(self, x):
        x = self.stage3(self.stage2(self.stage1(self.stem(x))))
        return self.classifier(self.pool(x).flatten(1))

model = CIFARResNet()
parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
for name, module in model.named_children():
    print(f'{name}: {sum(p.numel() for p in module.parameters() if p.requires_grad):,}')
print(f'Trainable parameters: {parameter_count:,}')
assert parameter_count <= 5_000_000, f'Parameter budget exceeded: {parameter_count:,}'
with torch.no_grad():
    model.eval()
    assert model(torch.zeros(2, 3, 32, 32)).shape == (2, 10)

## Train for 150 epochs
Cross entropy with SGD (learning rate 0.1, momentum 0.9, weight decay 5e-4), batch size 128, and cosine decay stepped once after each epoch. Training uses padded random crops and horizontal flips. Validation uses evaluation mode and deterministic preprocessing. Save the best validation checkpoint and per-epoch metrics in `/kaggle/working`.



In [ ]:
# Fresh weights for the full run; the diagnostic model is not reused.
seed_everything(SEED)
train_loader.generator.manual_seed(SEED)
model = CIFARResNet().to(device)
parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert parameter_count <= 5_000_000, f'Parameter budget exceeded: {parameter_count:,}'
print(f'Trainable parameters: {parameter_count:,}')
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(
    model.parameters(), lr=LEARNING_RATE, momentum=0.9, weight_decay=5e-4,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


def train_epoch():
    model.train()
    loss_sum, correct, count = 0.0, 0, 0
    for images, targets in train_loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, targets)
        assert torch.isfinite(loss), 'Non-finite training loss'
        loss.backward()
        optimizer.step()
        loss_sum += loss.detach().item() * targets.size(0)
        correct += (logits.detach().argmax(1) == targets).sum().item()
        count += targets.size(0)
    return loss_sum / count, correct / count


@torch.inference_mode()
def evaluate():
    model.eval()
    loss_sum, correct, count = 0.0, 0, 0
    for images, targets in val_loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        logits = model(images)
        loss_sum += criterion(logits, targets).item() * targets.size(0)
        correct += (logits.argmax(1) == targets).sum().item()
        count += targets.size(0)
    return loss_sum / count, correct / count


best_accuracy = -1.0
best_epoch = 0
history = []
for epoch in range(1, EPOCHS + 1):
    learning_rate = optimizer.param_groups[0]['lr']
    train_loss, train_accuracy = train_epoch()
    val_loss, val_accuracy = evaluate()
    scheduler.step()
    metrics = dict(epoch=epoch, learning_rate=learning_rate,
                   train_loss=train_loss, train_accuracy=train_accuracy,
                   val_loss=val_loss, val_accuracy=val_accuracy)
    history.append(metrics)
    if val_accuracy > best_accuracy:
        best_accuracy, best_epoch = val_accuracy, epoch
        torch.save(dict(
            model_state=model.state_dict(), epoch=epoch,
            val_accuracy=val_accuracy, parameter_count=parameter_count,
            classes=classes, seed=SEED,
            train_indices=train_indices.tolist(), val_indices=val_indices.tolist(),
            normalization=dict(mean=MEAN, std=STD),
        ), WORK_DIR / 'best_resnet.pt')
    # Flush history every epoch so metrics survive an interrupted Kaggle session.
    with (WORK_DIR / 'history.csv').open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=metrics.keys())
        writer.writeheader()
        writer.writerows(history)
    print(f'Epoch {epoch:03d}/{EPOCHS}: lr={learning_rate:.6f}, '
          f'train loss={train_loss:.4f}, train acc={train_accuracy:.2%}, '
          f'val loss={val_loss:.4f}, val acc={val_accuracy:.2%}, '
          f'best={best_accuracy:.2%}', flush=True)

# Leave the best model loaded for subsequent evaluation or test prediction.
checkpoint = torch.load(WORK_DIR / 'best_resnet.pt', map_location=device, weights_only=True)
model.load_state_dict(checkpoint['model_state'])
model.eval()
print(f'Best validation accuracy: {best_accuracy:.2%} (epoch {best_epoch})')
print(f'Trainable parameters: {parameter_count:,}')
print('Saved:', WORK_DIR / 'best_resnet.pt', WORK_DIR / 'history.csv')


## Generate the Kaggle submission
Run after training, or after running the setup and model definition cells with an existing `best_resnet.pt`. This section predicts every test image in sample submission order and writes `/kaggle/working/submission.csv` with exactly `id,label` columns and integer labels 0–9. The best checkpoint supplies the class mapping and deterministic normalization.


In [ ]:
# Reload the best checkpoint, rather than using the last training epoch.
checkpoint = torch.load(WORK_DIR / 'best_resnet.pt', map_location=device, weights_only=True)
model = CIFARResNet().to(device)
model.load_state_dict(checkpoint['model_state'])
model.eval()
assert sum(p.numel() for p in model.parameters()) <= 5_000_000

# Convert internal classifier positions back to official CIFAR-10 indices.
CIFAR10_CLASSES = (
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck',
)
def official_class_indices(saved_classes):
    names = [str(name) for name in saved_classes]
    if set(names) == {str(i) for i in range(10)}:
        return [int(name) for name in names]
    if set(names) == set(CIFAR10_CLASSES):
        return [CIFAR10_CLASSES.index(name) for name in names]
    raise ValueError(f'Unknown class mapping: {names}; provide the official mapping explicitly.')

class_indices = official_class_indices(checkpoint['classes'])
assert sorted(class_indices) == list(range(10))
normalization = checkpoint['normalization']
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(normalization['mean'], normalization['std']),
])

SAMPLE_SUBMISSION_CSV = INPUT_ROOT / 'sample_submission.csv'
TEST_IMAGE_DIR = INPUT_ROOT / 'test_images'
assert SAMPLE_SUBMISSION_CSV.is_file(), SAMPLE_SUBMISSION_CSV
assert TEST_IMAGE_DIR.is_dir(), TEST_IMAGE_DIR
with SAMPLE_SUBMISSION_CSV.open(newline='') as f:
    reader = csv.DictReader(f)
    assert reader.fieldnames == ['id', 'label'], 'Expected sample columns: id,label'
    sample_rows = list(reader)
test_ids = [row['id'] for row in sample_rows]
assert test_ids and all(image_id.strip() for image_id in test_ids), 'Empty test IDs'
assert len(set(test_ids)) == len(test_ids), 'Duplicate sample IDs'

def test_image_key(value):
    # Match IDs such as 1, test_00001, or test_00001.png without changing output IDs.
    stem = Path(str(value).strip()).stem
    if stem.startswith('test_'):
        stem = stem[len('test_'):]
    return str(int(stem)) if stem.isdigit() else stem

def ordered_test_paths(image_dir, ids):
    paths = sorted(p for p in image_dir.rglob('*')
                   if p.is_file() and p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.bmp'})
    by_key = {}
    for path in paths:
        key = test_image_key(path.name)
        if key in by_key:
            raise ValueError(f'Ambiguous test image ID: {by_key[key]} and {path}')
        by_key[key] = path
    keys = [test_image_key(image_id) for image_id in ids]
    assert len(set(keys)) == len(keys), 'Sample IDs resolve to duplicate images'
    missing = [image_id for image_id, key in zip(ids, keys) if key not in by_key]
    assert not missing, f'Missing test images: {missing[:5]}'
    assert set(keys) == set(by_key), 'Sample submission does not cover every test image'
    return [by_key[key] for key in keys]

test_paths = ordered_test_paths(TEST_IMAGE_DIR, test_ids)
class TestImages(Dataset):
    def __len__(self):
        return len(test_paths)
    def __getitem__(self, index):
        with Image.open(test_paths[index]) as image:
            image = image.convert('RGB')
            assert image.size == (32, 32), f'Unexpected test image size: {image.size}'
            return test_transform(image)

test_loader = DataLoader(
    TestImages(), batch_size=BATCH_SIZE, shuffle=False, drop_last=False,
    num_workers=NUM_WORKERS, pin_memory=device.type == 'cuda',
)
predictions = []
with torch.inference_mode():
    for images in test_loader:
        logits = model(images.to(device, non_blocking=True))
        assert logits.shape[1] == 10 and torch.isfinite(logits).all(), 'Invalid test logits'
        predictions.extend(class_indices[index] for index in logits.argmax(1).cpu().tolist())
assert len(predictions) == len(test_ids) == len(test_paths)
assert all(type(label) is int and 0 <= label <= 9 for label in predictions)

submission_path = WORK_DIR / 'submission.csv'
with submission_path.open('w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'label'])
    writer.writerows(zip(test_ids, predictions))

# Reopen the saved file to verify schema, order, coverage, and integer label encoding.
with submission_path.open(newline='') as f:
    reader = csv.DictReader(f)
    assert reader.fieldnames == ['id', 'label']
    submitted_rows = list(reader)
assert [row['id'] for row in submitted_rows] == test_ids
assert [row['label'] for row in submitted_rows] == [str(label) for label in predictions]
print(f'Saved {len(predictions):,} predictions to {submission_path}')
print('Preview:', submitted_rows[:5])
print('Download submission.csv from Kaggle outputs and upload it to the competition page.')
